# 02b – Làm sạch nhẹ và ẩn danh hóa bộ Synthea (Bronze → Silver)

**Chủ sở hữu: Thành viên 5** (Người 1 chuẩn bị bản đầu để bàn giao; Thành viên 5 duyệt và chịu trách nhiệm từ đây). Mục đích: đọc `patients.csv`, `encounters.csv`, `providers.csv`, `organizations.csv` trong `data/raw/`, ghi các file `*_anon.csv` của Synthea trong `data/anonymized/`. Minh chứng cho mục 3.3.2 và 3.5.

**Nguyên tắc**
- Notebook này là nơi duy nhất ghi các file Synthea trong `data/anonymized/`. Phần ER thuộc notebook `02a`.
- Muối, hàm băm, ngưỡng k, mốc tính tuổi và nhóm tuổi nằm trong `src/anon_common.py` (Người 1 sở hữu), không định nghĩa lại ở đây.
- Làm sạch không xóa dòng: ngoại lệ chỉ được gắn cờ `is_*`, ETL (03, 04) quyết định lọc. Nhờ vậy khóa ngoại không gãy.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
                            if (p / 'src' / 'anon_common.py').exists()) / 'src'))
import numpy as np
import pandas as pd
from anon_common import (ROOT, RAW, ANON, K_MIN, REF_DATE, AGE_BINS, AGE_LABELS,
                         get_salt, pseudonymize, Auditor, class_sizes, k_report, scan_leftovers)

SALT = get_salt()                      # dừng nếu thiếu, còn giá trị mẫu hoặc ngắn hơn 16 ký tự
audit = Auditor('02b_clean_anonymize_synthea')

## 2. Bảng kê trường định danh (bộ Synthea)

In [ ]:
inventory_syn = pd.DataFrame([
    ('Synthea', 'patients.Id, encounters.PATIENT', 'Trực tiếp', 'Băm HMAC-SHA256 có muối (cùng hàm với 02a)', 'Giữ được khóa nối giữa hai bảng'),
    ('Synthea', 'SSN, DRIVERS, PASSPORT', 'Trực tiếp', 'Bỏ', 'Số giấy tờ'),
    ('Synthea', 'PREFIX, FIRST, LAST, SUFFIX, MAIDEN', 'Trực tiếp', 'Bỏ', 'Họ tên'),
    ('Synthea', 'patients: ADDRESS, CITY, COUNTY, ZIP, LAT, LON, BIRTHPLACE', 'Trực tiếp / gián tiếp', 'Bỏ', 'Địa chỉ và tọa độ bệnh nhân (Safe Harbor yêu cầu bỏ mức nhỏ hơn bang)'),
    ('Synthea', 'patients / providers / organizations: STATE', 'Hằng số', 'Bỏ', 'Toàn bộ là Massachusetts (MA), không mang thông tin phân tích'),
    ('Synthea', 'BIRTHDATE', 'Gián tiếp', 'Bỏ; thay bằng age_group', 'Ngày sinh đầy đủ là định danh'),
    ('Synthea', 'DEATHDATE', 'Gián tiếp', 'Bỏ; thay bằng is_deceased', 'Ngày mất là định danh'),
    ('Synthea', 'RACE, ETHNICITY, GENDER, MARITAL', 'Gián tiếp', 'Gộp nhóm hiếm, ẩn cục bộ nếu cần', 'Mẫu chỉ 1.163 người nên nhóm nhỏ dễ lộ'),
    ('Synthea', 'providers: NAME, ADDRESS, ZIP, LAT, LON', 'Trực tiếp', 'Bỏ', 'Tên và địa chỉ bác sĩ'),
    ('Synthea', 'providers.CITY, organizations.CITY', 'Địa chỉ cơ sở (không phải của bệnh nhân)', 'Giữ ở mức thành phố', 'Dim_Hospital cần để nhóm cơ sở; không liên quan đến nơi ở của bệnh nhân'),
    ('Synthea', 'organizations: ADDRESS, PHONE, ZIP, LAT, LON', 'Địa chỉ cơ sở', 'Bỏ', 'Không cần cho phân tích'),
    ('Synthea', 'organizations.REVENUE', 'Không dùng được', 'Bỏ', 'Toàn bằng 0'),
], columns=['source', 'field', 'category', 'treatment', 'reason'])
inventory_syn.to_csv(ANON / 'identifier_inventory_synthea.csv', index=False)
inventory_syn

## 3. Bệnh nhân → `patients_anon.csv`

Tuổi tính tại ngày mất (nếu đã mất) hoặc tại mốc `REF_DATE` (31/12/2021). Sau khi gộp nhóm, các nhóm tương đương còn nhỏ hơn k bị **ẩn cục bộ** (đổi giá trị thuộc tính thành `Suppressed`) thay vì xóa bệnh nhân, để không làm mất khóa ngoại.

In [ ]:
pat = audit.read_csv('patients.csv')
birth = pd.to_datetime(pat['BIRTHDATE'])
end = pd.to_datetime(pat['DEATHDATE']).fillna(REF_DATE).clip(upper=REF_DATE)
age = (end - birth).dt.days // 365.25

pat_anon = pd.DataFrame({
    'patient_hash': pat['Id'].map(pseudonymize),
    'gender': pat['GENDER'],
    'age_group': pd.cut(age, AGE_BINS, right=False, labels=AGE_LABELS).astype(str),
    'race': pat['RACE'].where(pat['RACE'].isin(['white', 'black', 'asian']), 'other'),   # gộp hawaiian, native, other
    'ethnicity': pat['ETHNICITY'],
    'marital_status': pat['MARITAL'].fillna('Unknown'),
    'is_deceased': pat['DEATHDATE'].notna(),
    'healthcare_expenses_usd': pat['HEALTHCARE_EXPENSES'],
    'healthcare_coverage_usd': pat['HEALTHCARE_COVERAGE'],
})
assert pat_anon['patient_hash'].is_unique

QI_PAT = ['age_group', 'gender', 'race', 'ethnicity']
suppression_log = []
for col in ['race', 'ethnicity', 'age_group']:          # thuộc tính ít giá trị phân tích nhất bị ẩn trước
    small = class_sizes(pat_anon, QI_PAT) < K_MIN
    if not small.any():
        break
    pat_anon.loc[small, col] = 'Suppressed'
    suppression_log.append({'column': col, 'cells_suppressed': int(small.sum())})
print(suppression_log)

audit.record['dropped_columns']['patients'] = ['Id', 'BIRTHDATE', 'DEATHDATE', 'SSN', 'DRIVERS', 'PASSPORT', 'PREFIX', 'FIRST',
                                               'LAST', 'SUFFIX', 'MAIDEN', 'BIRTHPLACE', 'ADDRESS', 'CITY', 'STATE', 'COUNTY', 'ZIP', 'LAT', 'LON']
audit.write_csv(pat_anon, 'patients_anon.csv')

`marital_status` không nằm trong bộ thuộc tính được bảo đảm k ≥ 5 (thêm vào sẽ phải ẩn quá nhiều ô); rủi ro này được nêu ở báo cáo mục 3.5.4.

## 4. Lượt khám → `encounters_anon.csv`

Làm sạch không xóa dòng, chỉ gắn cờ:
- `is_in_analysis_period`: bắt đầu từ năm 2011 (trước đó thưa, lệch).
- `is_cost_outlier`: chi phí vượt P99 của riêng từng loại lượt khám, tính trong giai đoạn phân tích và bỏ các dòng chi phí 0. (Đã thử IQR trên thang tuyến tính và trên log(1 + chi phí) nhưng bỏ: IQR tuyến tính gắn cờ 21% số lượt; IQR trên log(1 + chi phí) tính riêng từng loại có ngưỡng trên từ khoảng 0,5 đến 30 triệu USD ở năm loại, nên không bắt được ca cực đoan nào, riêng wellness bị gắn cờ nhầm 1.574 lượt giá rẻ hợp lệ.)
- `is_zero_cost`: tổng chi phí bằng 0.
- `is_duration_outlier`: thời lượng > 24 giờ với loại không nội trú; nội trú > 30 ngày.
- `reason_description` trống được điền `Not recorded` và có cờ `has_reason`.

In [ ]:
enc = audit.read_csv('encounters.csv', low_memory=False)
start = pd.to_datetime(enc['START'], utc=True)
stop = pd.to_datetime(enc['STOP'], utc=True)
duration_min = (stop - start).dt.total_seconds() / 60

in_period = start.dt.year >= 2011
cost = enc['TOTAL_CLAIM_COST']
cost_outlier = pd.Series(False, index=enc.index)
for cls, idx in enc.groupby('ENCOUNTERCLASS').groups.items():
    ref = cost.loc[[i for i in idx if in_period.loc[i] and cost.loc[i] > 0]]
    cost_outlier.loc[idx] = cost.loc[idx] > ref.quantile(0.99)

dur_limit = np.where(enc['ENCOUNTERCLASS'].eq('inpatient'), 30 * 1440, 1440)

enc_anon = pd.DataFrame({
    'encounter_id': enc['Id'],
    'patient_hash': enc['PATIENT'].map(pseudonymize),
    'organization_id': enc['ORGANIZATION'],
    'provider_id': enc['PROVIDER'],
    'payer_id': enc['PAYER'],
    'encounter_class': enc['ENCOUNTERCLASS'],
    'encounter_code': enc['CODE'],
    'encounter_description': enc['DESCRIPTION'],
    'start_datetime': start.dt.strftime('%Y-%m-%d %H:%M:%S'),
    'stop_datetime': stop.dt.strftime('%Y-%m-%d %H:%M:%S'),
    'duration_min': duration_min.round(1),
    'base_cost_usd': enc['BASE_ENCOUNTER_COST'],
    'total_cost_usd': enc['TOTAL_CLAIM_COST'],
    'payer_coverage_usd': enc['PAYER_COVERAGE'],
    'reason_description': enc['REASONDESCRIPTION'].fillna('Not recorded'),
    'has_reason': enc['REASONDESCRIPTION'].notna(),
    'is_in_analysis_period': in_period,
    'is_cost_outlier': cost_outlier,
    'is_zero_cost': enc['TOTAL_CLAIM_COST'].eq(0),
    'is_duration_outlier': duration_min > dur_limit,
})
assert enc_anon['patient_hash'].isin(pat_anon['patient_hash']).all()     # mọi khóa nối vẫn khớp sau khi băm
audit.record['dropped_columns']['encounters'] = ['Id (đổi tên)', 'PATIENT (băm)', 'REASONCODE']
audit.write_csv(enc_anon, 'encounters_anon.csv')
enc_anon[['is_in_analysis_period', 'is_cost_outlier', 'is_zero_cost', 'is_duration_outlier', 'has_reason']].sum()

## 5. Nhà cung cấp và tổ chức

In [ ]:
prov = audit.read_csv('providers.csv')
prov_anon = pd.DataFrame({
    'provider_id': prov['Id'], 'organization_id': prov['ORGANIZATION'], 'gender': prov['GENDER'],
    'specialty': prov['SPECIALITY'], 'city': prov['CITY'], 'utilization': prov['UTILIZATION'],
})
audit.record['dropped_columns']['providers'] = ['NAME', 'ADDRESS', 'STATE', 'ZIP', 'LAT', 'LON']
audit.write_csv(prov_anon, 'providers_anon.csv')

org = audit.read_csv('organizations.csv')
assert (org['REVENUE'] == 0).all()     # xác nhận REVENUE vô dụng trước khi bỏ
org_anon = pd.DataFrame({
    'organization_id': org['Id'],
    'organization_name': org['NAME'].str.replace(r'\s+', ' ', regex=True).str.strip(),
    'city': org['CITY'], 'utilization': org['UTILIZATION'],
})
audit.record['dropped_columns']['organizations'] = ['ADDRESS', 'STATE', 'ZIP', 'LAT', 'LON', 'PHONE', 'REVENUE']
audit.write_csv(org_anon, 'organizations_anon.csv')

## 6. Kiểm tra k-anonymity (Synthea)

In [ ]:
rows = [
    k_report(pat_anon, QI_PAT, 'Synthea', 'A: nhân khẩu học sau ẩn cục bộ (ràng buộc)'),
    k_report(pat_anon, QI_PAT + ['marital_status'], 'Synthea', 'B: A + tình trạng hôn nhân'),
]
k_table = pd.DataFrame(rows)
k_table.to_csv(ANON / 'k_anonymity_report_synthea.csv', index=False)
audit.record['k_anonymity'] = rows
audit.record['suppression'] = suppression_log
assert k_table.loc[k_table['qi_set'].str.startswith('A'), 'k_min'].min() >= K_MIN, 'k-anonymity chưa đạt'
k_table.drop(columns='qi_columns')

## 7. Quét sót định danh trong đầu ra Synthea

In [ ]:
scan_leftovers(
    {'patients_anon.csv': pat_anon, 'encounters_anon.csv': enc_anon,
     'providers_anon.csv': prov_anon, 'organizations_anon.csv': org_anon},
    raw_ids=set(pat['Id']),
    forbidden_cols={'SSN', 'DRIVERS', 'PASSPORT', 'FIRST', 'LAST', 'ADDRESS', 'LAT', 'LON', 'BIRTHDATE', 'NAME'})
print('Quét sót: không còn mẫu SSN, mã gốc hay cột định danh trong 4 file đầu ra Synthea.')

## 8. Nhật ký chạy

Ghi vào `anonymization_audit_synthea.jsonl` (file riêng của notebook này); người chạy ghi bằng mã thành viên qua biến `RUN_BY`.

In [ ]:
audit.save('anonymization_audit_synthea.jsonl')
print('Đã ghi:', *sorted(p.name for p in ANON.glob('*') if p.name.endswith('_anon.csv') and 'er_' not in p.name), sep='\n- ')